In [36]:
import os
import openai
import requests
import uuid
from pathlib import Path
from sentence_transformers import SentenceTransformer

/opt/homebrew/Cellar/jupyterlab/4.3.5/libexec/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
QDRANT_URL = "http://localhost:6333"
COLLECTION_NAME = "my-collection"
VECTOR_DIM = 384

### Load Documents

In [30]:
DOCS_FOLDER = Path("docs")
all_text_chunks = []

def chunk_text(text, chunk_size=500, overlap=50):
    """
    Split chunks into texts of size 'chunk_size' with an overlap of `overlap` characters.
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - overlap
    return chunks

for md_file in DOCS_FOLDER.glob("*.md"):
    with open(md_file, "r", encoding="utf-8") as f:
        file_content = f.read()
        for c in chunk_text(file_content):
            all_text_chunks.append({
                "text":c,
                "source":md_file.name
            })

all_text_chunks

[{'text': '# Frequently Asked Questions\n\n**Q**: What is Retrieval-Augmented Generation (RAG)?  \n**A**: RAG is a process that uses a language model (like ChatGPT) in combination with an external knowledge base. The knowledge base is typically indexed in a vector database, so that chunks of text can be retrieved based on semantic similarity.\n\n**Q**: Why use a vector database?  \n**A**: A vector database allows you to find semantically similar documents or text passages quickly, enabling more accurate context ',
  'source': 'faq.md'},
 {'text': ' passages quickly, enabling more accurate context retrieval for your queries.\n\n**Q**: Do I need advanced infrastructure?  \n**A**: For small projects, running Qdrant, Weaviate, or Milvus locally via Docker is usually enough. For larger projects, you might need more robust cloud or on-prem deployments.\n',
  'source': 'faq.md'},
 {'text': '# Welcome to Our Sample Knowledge Base\n\nThis document provides an introduction to building a Retrieva

### Create Qdrant Collection

In [33]:
def create_collection():
    """
    We specify the vector size for Qdrant. For text-embedding-ada-002, vector
    size is 1536. Distance can be "Cosine", "Euclid" or "Dot"
    """
    payload = {
        "name": "my-collection",
        "vector_size": VECTOR_DIM,
        "distance": "Cosine"
    }
    response = requests.put(f"{QDRANT_URL}/collections/{COLLECTION_NAME}", json=payload)
    if response.status_code not in (200, 201):
        print(f"Error creating collection:", response.text)
    else:
        print(f"Collection created or already exisits")

create_collection()

Collection created or already exisits


### Embedding and Upsert

In [38]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def get_local_embedding(text):
    """
    Generate a local embedding using Sentence Transformers, returning a list float.
    """
    # model.encode returns a NumPy array; convert to list for JSON serialization
    embedding = model.encode(text).tolist()
    print(f"{embedding}")
    return embedding

def upsert_point_to_qdrant(vector, payload):
    point_id = str(uuid.uuid4())
    data = {
        "points": [
            {
                "id": point_id,
                "vector": vector,
                "payload": payload
            }
        ]
    }
    url = f"{QDRANT_URL}/collections/{COLLECTION_NAME}/points?wait=true"
    response = requests.put(url, json=data)
    if response.status_code not in (200, 201):
        print(f"Error upserting point", response.text)

def ingest_data():
    for chunk_info in all_text_chunks[:1]:
        text_chunk = chunk_info["text"]
        print(text_chunk)
        source_file = chunk_info["source"]
        print(source)
        vector = get_embedding(text_chunk)
        print(vector)
        metadata = {
            "text": text_chunk,
            "source": source_file
        }
        upsert_point_to_qdrant(vector, metadata)

ingest_data()

Text selected for embedding: # Frequently Asked Questions

**Q**: What is Retrieval-Augmented Generation (RAG)?  
**A**: RAG is a process that uses a language model (like ChatGPT) in combination with an external knowledge base. The knowledge base is typically indexed in a vector database, so that chunks of text can be retrieved based on semantic similarity.

**Q**: Why use a vector database?  
**A**: A vector database allows you to find semantically similar documents or text passages quickly, enabling more accurate context 
Error getting embedding: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
Point Id : 6a4b9925-f59a-4097-92d1-84e9b95a67a2
Error upserting point {"status":{"error":"Format error in JSON body: data did not matc

  Using cached regex-2024.11.6-cp313-cp313-macosx_11_0_arm64.whl.metadata (40 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 MB 30.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 22.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 28.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 31.8 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.8/24.8 MB 29.4 MB/s eta 0:00:0000:0100:01
Using cached regex-2024.11.6-cp313-cp313-macosx_11_0_arm64.whl (284 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 14.6 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.3.1 -> 25.0
[notice] To update, run: python -m pip